# Batch B: 4 Medium Datasets (Colab Pro)

**Datasets**: TREC-COVID (171K) + Touché-2020 (382K) + Quora (523K) + CQADupStack (12 subforums)

**CQADupStack**: 12 separate subforums, each evaluated independently, report average.

**Time**: ~8-10 hours on T4. Checkpoint every 5000 docs. Disconnect-safe.

In [1]:
!pip install -q beir sentence-transformers rank-bm25 numpy pytrec-eval-terrier
!nvidia-smi | head -5 || echo 'No GPU'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 41.2 MB/s eta 0:00:00
Sat Mar 28 06:35:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [2]:
import json
import os
import re
import time
from collections import defaultdict

import numpy as np
import pytrec_eval
import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Device: cuda
GPU: Tesla T4


In [3]:
# ── Fusion + eval functions ──

def _norm(results):
    if not results:
        return {}
    vals = [s for _, s in results]
    mn, mx = min(vals), max(vals)
    rng = mx - mn if mx > mn else 1.0
    return {did: (s - mn) / rng for did, s in results}

def simple_rrf(b, d, k=5, bw=1.0, dw=1.2):
    scores = defaultdict(float)
    for rank, (did, _) in enumerate(b):
        scores[did] += bw / (k + rank + 1)
    for rank, (did, _) in enumerate(d):
        scores[did] += dw / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def riverbed_only(b, d, bw=0.8, dw=1.4):
    b_n, d_n = _norm(b), _norm(d)
    all_docs = set(b_n) | set(d_n)
    tw = bw + dw
    final = {did: (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw for did in all_docs}
    return sorted(final.items(), key=lambda x: x[1], reverse=True)

def rt_full(b, d, k_low=3, k_high=10, top_n=20, boost_max=1.2, score_w=0.5, bw=0.8, dw=1.4):
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    agreement = len(b_set & d_set) / len(union) if union else 0.0
    tension = 1.0 - agreement
    adaptive_k = max(1, int(k_low + (k_high - k_low) * tension))
    boost = 1.0 + (boost_max - 1.0) * agreement
    rrf_scores = defaultdict(float)
    presence = defaultdict(int)
    for rank, (did, _) in enumerate(b):
        rrf_scores[did] += bw / (adaptive_k + rank + 1)
        presence[did] += 1
    for rank, (did, _) in enumerate(d):
        rrf_scores[did] += dw / (adaptive_k + rank + 1)
        presence[did] += 1
    for did in rrf_scores:
        if presence[did] >= 2:
            rrf_scores[did] *= boost
    b_n, d_n = _norm(b), _norm(d)
    rv = list(rrf_scores.values())
    r_mn, r_mx = min(rv), max(rv)
    r_rng = r_mx - r_mn if r_mx > r_mn else 1.0
    tw = bw + dw
    all_docs = set(rrf_scores) | set(b_n) | set(d_n)
    final = {}
    for did in all_docs:
        r = (rrf_scores.get(did, 0) - r_mn) / r_rng
        s = (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        final[did] = (1 - score_w) * r + score_w * s
    return sorted(final.items(), key=lambda x: x[1], reverse=True)

STRATEGIES = {
    "dense_only": lambda b, d: d,
    "rrf": lambda b, d: simple_rrf(b, d),
    "riverbed": lambda b, d: riverbed_only(b, d),
    "rt_full": lambda b, d: rt_full(b, d),
}
METRICS = {"ndcg_cut_10", "recall_100", "map"}

def evaluate_official(qrels, run_dict):
    qrels_int = {qid: {did: int(rel) for did, rel in rels.items()} for qid, rels in qrels.items()}
    evaluator = pytrec_eval.RelevanceEvaluator(qrels_int, METRICS)
    scores = evaluator.evaluate(run_dict)
    result = {}
    for metric in METRICS:
        vals = [scores[qid].get(metric, 0) for qid in scores]
        result[metric] = round(sum(vals) / len(vals), 6)
    return result

print("All functions ready")

All functions ready


In [ ]:
# ── Download datasets (local) + cache on Drive ──

from google.colab import drive
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/beir_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

BEIR_BASE = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets"
BASE_DIR = "datasets"
os.makedirs(BASE_DIR, exist_ok=True)

DATASETS_SIMPLE = {
    "trec-covid": f"{BEIR_BASE}/trec-covid.zip",
    "webis-touche2020": f"{BEIR_BASE}/webis-touche2020.zip",
    "quora": f"{BEIR_BASE}/quora.zip",
}

loaded_simple = {}
for ds_name, url in DATASETS_SIMPLE.items():
    print(f"\n{ds_name}...")
    data_path = os.path.join(BASE_DIR, ds_name)
    if not os.path.isdir(data_path):
        data_path = util.download_and_unzip(url, BASE_DIR)
    corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")
    loaded_simple[ds_name] = (corpus, queries, qrels)
    print(f"  {len(corpus)} docs, {len(queries)} queries")

# CQADupStack: 12 subforums
CQA_URL = f"{BEIR_BASE}/cqadupstack.zip"
CQA_FORUMS = [
    "android", "english", "gaming", "gis", "mathematica",
    "physics", "programmers", "stats", "tex",
    "unix", "webmasters", "wordpress",
]

cqa_path = os.path.join(BASE_DIR, "cqadupstack")
if not os.path.isdir(cqa_path):
    print("\nDownloading cqadupstack...")
    cqa_path = util.download_and_unzip(CQA_URL, BASE_DIR)

loaded_cqa = {}
total_cqa_docs = 0
for forum in CQA_FORUMS:
    forum_path = os.path.join(cqa_path, forum)
    corpus, queries, qrels = GenericDataLoader(forum_path).load(split="test")
    loaded_cqa[forum] = (corpus, queries, qrels)
    total_cqa_docs += len(corpus)
    print(f"  cqa/{forum}: {len(corpus)} docs, {len(queries)} queries")

print(f"\nCQA total: {total_cqa_docs} docs across {len(loaded_cqa)} subforums")

In [5]:
# ── Load model ──

MODEL_ID = "intfloat/e5-base-unsupervised"
PREFIX_Q = "query: "
PREFIX_D = "passage: "

print(f"Loading {MODEL_ID}...")
model = SentenceTransformer(MODEL_ID, device=DEVICE)
print(f"Loaded on {DEVICE}")

Loading intfloat/e5-base-unsupervised...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Loaded on cuda


In [ ]:
# ── Encode + BM25 + evaluate helper ──

def tokenize(text):
    return re.findall(r'\w+', text.lower())

def encode_corpus(corpus, ds_name):
    tag = MODEL_ID.replace('/', '_').replace('-', '_')
    cache_path = os.path.join(CACHE_DIR, f".cache_{ds_name}_{tag}.npz")
    ckpt_path = os.path.join(CACHE_DIR, f".cache_{ds_name}_{tag}.ckpt.npz")
    doc_id_list = list(corpus.keys())
    texts = [f"{PREFIX_D}{corpus[did].get('title', '')} {corpus[did].get('text', '')}".strip() for did in doc_id_list]
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        print(f"  Cache hit: {data['embs'].shape}")
        return data["embs"], doc_id_list
    start_idx = 0
    all_embs = []
    if os.path.exists(ckpt_path):
        ckpt = np.load(ckpt_path)
        start_idx = int(ckpt["done"])
        all_embs = [ckpt["embs"]]
        print(f"  Resuming from {start_idx}/{len(texts)}")
    batch_size = 256
    t0 = time.time()
    for i in range(start_idx, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embs = model.encode(batch, normalize_embeddings=True, show_progress_bar=False, batch_size=128)
        all_embs.append(embs)
        done = min(i + batch_size, len(texts))
        if done % 5000 < batch_size or done == len(texts):
            partial = np.vstack(all_embs)
            np.savez_compressed(ckpt_path, embs=partial, done=done)
            elapsed = time.time() - t0
            speed = (done - start_idx) / elapsed if elapsed > 0 else 0
            eta = (len(texts) - done) / speed if speed > 0 else 0
            print(f"  {done}/{len(texts)} ({speed:.0f} d/s, ETA {eta:.0f}s)")
    passage_embs = np.vstack(all_embs)
    np.savez_compressed(cache_path, embs=passage_embs)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    total = time.time() - t0
    print(f"  Done: {passage_embs.shape} in {total:.0f}s ({total/60:.1f}min)")
    return passage_embs, doc_id_list

def evaluate_dataset(ds_name, corpus, queries, qrels):
    print(f"  Encoding...")
    passage_embs, doc_id_list = encode_corpus(corpus, ds_name)
    print(f"  Building BM25...")
    doc_ids = list(corpus.keys())
    tokenized = [tokenize(f"{corpus[did].get('title','')} {corpus[did].get('text','')}") for did in doc_ids]
    bm25 = BM25Okapi(tokenized)
    def search_bm25(query, top_k=100):
        scores = bm25.get_scores(tokenize(query))
        top_idx = scores.argsort()[-top_k:][::-1]
        return [(doc_ids[i], float(scores[i])) for i in top_idx if scores[i] > 0]
    print(f"  Caching {len(queries)} queries...")
    cached = {}
    t0 = time.time()
    for qi, (qid, qt) in enumerate(queries.items()):
        bm25_res = search_bm25(qt, top_k=100)
        q_emb = model.encode([PREFIX_Q + qt], normalize_embeddings=True)
        sims = (passage_embs @ q_emb.T).flatten()
        idx = np.argsort(sims)[::-1][:100]
        dense_res = [(doc_id_list[i], float(sims[i])) for i in idx]
        cached[qid] = {"bm25": bm25_res, "dense": dense_res}
        if (qi + 1) % 100 == 0:
            elapsed = time.time() - t0
            speed = (qi + 1) / elapsed
            eta = (len(queries) - qi - 1) / speed
            print(f"    {qi+1}/{len(queries)} ({speed:.1f} q/s, ETA {eta:.0f}s)")
    print(f"  Cached in {time.time()-t0:.1f}s")
    ds_results = {}
    for strat_name, strat_fn in STRATEGIES.items():
        run_dict = {}
        for qid in queries:
            e = cached[qid]
            fused = strat_fn(e["bm25"], e["dense"])
            run_dict[qid] = {did: float(score) for did, score in fused[:100]}
        metrics = evaluate_official(qrels, run_dict)
        ds_results[strat_name] = metrics
        print(f"  {strat_name:<15} nDCG@10={metrics['ndcg_cut_10']:.4f}  MAP={metrics['map']:.4f}")
    best = max(ds_results.items(), key=lambda x: x[1]["ndcg_cut_10"])
    print(f"  >>> BEST: {best[0]} = {best[1]['ndcg_cut_10']:.4f}")
    del cached, bm25
    return ds_results

print("Helpers ready")

In [ ]:
# ── Run simple datasets ──

ALL_RESULTS = {}
for ds_name, (corpus, queries, qrels) in loaded_simple.items():
    print(f"\n{'=' * 60}")
    print(f"{ds_name.upper()} ({len(corpus)} docs, {len(queries)} queries)")
    print(f"{'=' * 60}")
    ALL_RESULTS[ds_name] = evaluate_dataset(ds_name, corpus, queries, qrels)
print("\nSimple datasets done")


TREC-COVID (171332 docs, 50 queries)
  Encoding...
  5120/171332 (37 d/s, ETA 4482s)
  10240/171332 (39 d/s, ETA 4079s)
  15104/171332 (40 d/s, ETA 3893s)
  20224/171332 (42 d/s, ETA 3589s)
  25088/171332 (43 d/s, ETA 3395s)
  30208/171332 (42 d/s, ETA 3377s)
  35072/171332 (40 d/s, ETA 3365s)
  40192/171332 (40 d/s, ETA 3317s)
  45056/171332 (39 d/s, ETA 3263s)
  50176/171332 (38 d/s, ETA 3186s)
  55040/171332 (37 d/s, ETA 3113s)
  60160/171332 (37 d/s, ETA 3016s)
  65024/171332 (36 d/s, ETA 2913s)
  70144/171332 (36 d/s, ETA 2794s)
  75008/171332 (36 d/s, ETA 2680s)
  80128/171332 (36 d/s, ETA 2547s)
  85248/171332 (36 d/s, ETA 2409s)
  90112/171332 (36 d/s, ETA 2266s)
  95232/171332 (36 d/s, ETA 2107s)
  100096/171332 (36 d/s, ETA 1959s)
  105216/171332 (37 d/s, ETA 1807s)
  110080/171332 (37 d/s, ETA 1665s)
  115200/171332 (37 d/s, ETA 1517s)
  120064/171332 (37 d/s, ETA 1380s)
  125184/171332 (37 d/s, ETA 1248s)
  130048/171332 (37 d/s, ETA 1123s)
  135168/171332 (37 d/s, ETA 990

In [ ]:
# ── Run CQADupStack (12 subforums → average) ──

cqa_results = {}
for forum, (corpus, queries, qrels) in loaded_cqa.items():
    print(f"\n{'=' * 60}")
    print(f"CQA/{forum.upper()} ({len(corpus)} docs, {len(queries)} queries)")
    print(f"{'=' * 60}")
    cqa_results[forum] = evaluate_dataset(f"cqa_{forum}", corpus, queries, qrels)

# Average across subforums (BEIR standard)
cqa_avg = {}
for strat in STRATEGIES:
    avg_metrics = {}
    for metric in METRICS:
        vals = [cqa_results[f][strat][metric] for f in cqa_results]
        avg_metrics[metric] = round(sum(vals) / len(vals), 6)
    cqa_avg[strat] = avg_metrics

ALL_RESULTS["cqadupstack"] = cqa_avg
print(f"\n{'=' * 60}")
print(f"CQADUPSTACK AVERAGE ({len(cqa_results)} subforums)")
print(f"{'=' * 60}")
for strat, m in cqa_avg.items():
    print(f"  {strat:<15} nDCG@10={m['ndcg_cut_10']:.4f}  MAP={m['map']:.4f}")
best = max(cqa_avg.items(), key=lambda x: x[1]["ndcg_cut_10"])
print(f"  >>> BEST: {best[0]} = {best[1]['ndcg_cut_10']:.4f}")

In [ ]:
# ── Summary (Batch A + B = 9 datasets) ──

BATCH_A = {
    "scifact": {"riverbed": 0.7576, "rt_full": 0.7557, "rrf": 0.7503, "dense_only": 0.7371},
    "nfcorpus": {"riverbed": 0.3633, "rt_full": 0.3666, "rrf": 0.3609, "dense_only": 0.3585},
    "arguana": {"riverbed": 0.3286, "rt_full": 0.3318, "rrf": 0.3347, "dense_only": 0.3174},
    "scidocs": {"riverbed": 0.2116, "rt_full": 0.2110, "rrf": 0.2056, "dense_only": 0.2110},
    "fiqa": {"riverbed": 0.4160, "rt_full": 0.4122, "rrf": 0.3962, "dense_only": 0.4008},
}

print("\n" + "=" * 80)
print("ALL 9 DATASETS — Confluence Fusion (pytrec_eval official)")
print("=" * 80)

all_ds = {**BATCH_A}
for ds_name in list(DATASETS_SIMPLE) + ["cqadupstack"]:
    r = ALL_RESULTS[ds_name]
    all_ds[ds_name] = {s: r[s]["ndcg_cut_10"] for s in STRATEGIES}

strats = list(STRATEGIES.keys())
header = f"{'Dataset':<20}" + "".join(f"{s:>14}" for s in strats) + f"{'BEST':>14}" + f"{'vs Dense':>10}"
print(header)
print("-" * len(header))

wins = 0
total = 0
for ds_name, scores in all_ds.items():
    best_strat = max(scores, key=scores.get)
    best_score = scores[best_strat]
    dense = scores["dense_only"]
    total += 1
    if best_score > dense:
        wins += 1
    row = f"{ds_name:<20}"
    for s in strats:
        marker = " *" if s == best_strat else "  "
        row += f"{scores[s]:>12.4f}{marker}"
    row += f"{best_strat:>14}"
    row += f"{best_score - dense:>+10.4f}"
    print(row)

print(f"\nConfluence beats Dense: {wins}/{total}")

In [ ]:
# ── Save ONE JSON ──

output = {
    "experiment": "batch_b_beir_medium",
    "model": {"name": "E5-base", "hf_id": MODEL_ID, "params": "110M"},
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU",
    "evaluator": "pytrec_eval 0.5.10 (official BEIR standard)",
    "submission_ready": True,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "datasets": {},
    "cqa_per_forum": {},
}
for ds_name, (corpus, queries, _) in loaded_simple.items():
    output["datasets"][ds_name] = {
        "corpus_size": len(corpus),
        "num_queries": len(queries),
        "results": ALL_RESULTS[ds_name],
    }
output["datasets"]["cqadupstack"] = {
    "num_subforums": len(loaded_cqa),
    "results": ALL_RESULTS["cqadupstack"],
}
for forum, (corpus, queries, _) in loaded_cqa.items():
    output["cqa_per_forum"][forum] = {
        "corpus_size": len(corpus),
        "num_queries": len(queries),
        "results": cqa_results[forum],
    }
print(json.dumps(output, indent=2))